# Experimento 2 — SVM com balanceamento utilizando 4 variáveis

Este experimento utiliza o algoritmo SVM em sua versão linear (`LinearSVC`) aplicado ao dataset de qualidade hídrica, mantendo as mesmas condições dos experimentos anteriores para permitir comparação direta entre os algoritmos.

O Experimento 2 introduz o parâmetro `class_weight="balanced"` como principal modificação em relação ao Experimento 1. Esse parâmetro ajusta automaticamente o peso das classes de acordo com sua frequência no dataset, dando maior importância às classes minoritárias.

As variáveis utilizadas como entrada foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Country`
- `Waterbody Type`

A variável alvo (`y`) utilizada para previsão foi:

- `conama_status`

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi feita com `train_test_split`, utilizando `random_state=42` e `stratify=y`.

Como o `SVC` tradicional apresentou custo computacional elevado para o tamanho do dataset, foi utilizado o `LinearSVC`, que é mais adequado para bases maiores e mantém a proposta de um modelo linear baseado em margem.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

print("Imports carregados com sucesso.")

Imports carregados com sucesso.


In [6]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        r"C:\Users\anaju\OneDrive\projetos_lucas\AquaSense-ML\amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path(r"C:\Users\anaju\OneDrive\projetos_lucas\AquaSense-ML\amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente local/VS Code detectado.
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [7]:
# PREPARAÇÃO PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [8]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [9]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 4)
Teste: (28280, 4)


In [10]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [11]:
# MODELO SVM COM BALANCEAMENTO
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=SEED,
                max_iter=5000
            )
        )
    ]
)

print("Iniciando treinamento do SVM...")

model.fit(X_train, y_train)

print("SVM balanceado treinado com sucesso.")

Iniciando treinamento do SVM...
SVM balanceado treinado com sucesso.


## Avaliação das Métricas de Treino

Antes da avaliação no conjunto de teste, foram calculadas as métricas no conjunto de treino. Essa etapa permite comparar o comportamento do modelo entre treino e teste, identificando possíveis sinais de overfitting ou underfitting.

In [12]:
# MÉTRICAS DE TREINO
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("\nTrain Precision:")
print(train_precision)

print("\nTrain Recall:")
print(train_recall)

print("\nTrain F1:")
print(train_f1)

print("\nTrain Confusion Matrix:")
print(train_confusion_matrix)



Train Accuracy:
0.6828472670373678

Train Precision:
0.6886608033304348

Train Recall:
0.6828472670373678

Train F1:
0.6450761280937389

Train Confusion Matrix:
[[ 4613     4  1490  1453]
 [ 6722    55  1402 13499]
 [  590     1   378   137]
 [ 9201    95  1282 72197]]


## Avaliação no Conjunto de Teste

Nesta etapa, o modelo é avaliado em dados que não foram utilizados durante o treinamento. A análise considera não apenas a acurácia, mas também precision, recall, F1-score e matriz de confusão, com atenção especial às classes minoritárias `Atenção` e `Crítica`.

In [13]:
# MÉTRICAS DE TESTE
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:")
print(test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.6821428571428572

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.22      0.60      0.32      1890
         Boa       0.44      0.00      0.01      5419
     Crítica       0.07      0.32      0.12       277
   Excelente       0.82      0.87      0.85     20694

    accuracy                           0.68     28280
   macro avg       0.39      0.45      0.32     28280
weighted avg       0.70      0.68      0.64     28280


Confusion Matrix:
[[ 1140     1   379   370]
 [ 1599    21   377  3422]
 [  153     0    88    36]
 [ 2284    26   342 18042]]


## Resultados Obtidos — Experimento 2

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.682
```

O modelo SVM balanceado apresentou desempenho diferente do Experimento 1. A acurácia global caiu em relação ao SVM sem balanceamento, mas isso não significa necessariamente piora do modelo, pois o objetivo principal deste experimento era avaliar se o balanceamento ajudaria na identificação das classes minoritárias.

### Comparação entre algoritmos com balanceamento

| Métrica | RF c/ bal | LightGBM c/ bal | Regressão Logística c/ bal | SVM c/ bal |
|---|---:|---:|---:|---:|
| Accuracy teste | 0.710 | 0.599 | 0.644 | 0.682 |
| Precision Atenção | — | — | 0.21 | 0.22 |
| Precision Boa | — | — | 0.33 | 0.44 |
| Precision Crítica | — | — | 0.08 | 0.07 |
| Precision Excelente | — | — | 0.86 | 0.82 |
| Recall Atenção | — | — | 0.46 | 0.60 |
| Recall Boa | — | — | 0.19 | 0.00 |
| Recall Crítica | — | — | 0.63 | 0.32 |
| Recall Excelente | — | — | 0.78 | 0.87 |
| F1 Atenção | — | — | 0.28 | 0.32 |
| F1 Boa | — | — | 0.24 | 0.01 |
| F1 Crítica | — | — | 0.14 | 0.12 |
| F1 Excelente | — | — | 0.82 | 0.85 |
| Macro avg F1 | — | — | 0.37 | 0.32 |
| Weighted avg F1 | — | 0.58 | 0.66 | 0.64 |

### Análise das métricas

O SVM balanceado apresentou acurácia de 0.682, ficando abaixo apenas do Random Forest balanceado, que manteve aproximadamente 0.710 de accuracy. Ainda assim, o SVM superou o LightGBM balanceado (0.599) e também a Regressão Logística balanceada (0.644).

Entretanto, como o dataset AquaSense é altamente desbalanceado, a acurácia não pode ser utilizada como única métrica de avaliação. O principal objetivo do balanceamento foi reduzir o viés da classe majoritária (`Excelente`) e aumentar a capacidade dos modelos em detectar classes críticas ambientalmente relevantes.

A principal mudança ocorreu no comportamento das classes minoritárias. A classe `Atenção` teve recall de 0.60, indicando que o SVM passou a identificar uma quantidade relevante dessas amostras. Esse resultado foi superior ao da Regressão Logística, que obteve recall de 0.46 para a mesma classe.

Na classe `Crítica`, o SVM alcançou recall de 0.32. Esse valor representa uma melhora importante em relação ao SVM sem balanceamento, que não conseguia identificar corretamente essa classe. No entanto, a Regressão Logística balanceada apresentou recall maior para `Crítica`, com 0.63. Isso mostra que a Regressão Logística foi mais sensível para detectar casos críticos, mesmo gerando muitos falsos positivos.

A precision da classe `Crítica` permaneceu baixa no SVM, com 0.07. Isso significa que, apesar de o modelo passar a encontrar mais amostras críticas, muitas das previsões feitas como `Crítica` estavam incorretas. Esse comportamento é esperado em modelos balanceados, pois o algoritmo passa a arriscar mais para não ignorar classes minoritárias.

A classe `Boa` apresentou o pior resultado do SVM balanceado. O recall foi praticamente nulo, indicando que o modelo quase não conseguiu identificar corretamente essa classe. Isso mostra que, mesmo com balanceamento, o SVM teve dificuldade em separar classes intermediárias.

Já a classe `Excelente` continuou com desempenho alto, apresentando recall de 0.87 e F1-score de 0.85. Isso indica que o modelo ainda mantém forte capacidade de reconhecer a classe majoritária, embora menos enviesado do que no experimento sem balanceamento.

### Análise da Matriz de Confusão

A matriz de confusão mostra que o SVM balanceado passou a distribuir melhor suas previsões entre `Atenção`, `Crítica` e `Excelente`, reduzindo parcialmente o colapso na classe majoritária.

Entretanto, ainda houve forte confusão entre classes intermediárias. Muitas amostras reais de `Boa` foram classificadas como `Excelente`, e muitas amostras de `Excelente` também foram redistribuídas para `Atenção` e `Crítica`. Isso explica a queda na acurácia e a baixa precision nas classes minoritárias.

O ponto positivo é que o modelo passou a detectar parte das amostras `Críticas`, o que é essencial para o contexto ambiental do AquaSense. O ponto negativo é que essa melhora veio acompanhada de aumento nos falsos alarmes.

### Conclusão

O Experimento 2 demonstrou que o uso de `class_weight="balanced"` alterou significativamente o comportamento do SVM. Embora a acurácia tenha permanecido abaixo do Random Forest balanceado, o modelo passou a identificar melhor classes minoritárias quando comparado ao SVM sem balanceamento.

Comparando os quatro algoritmos testados até o momento:

- O Random Forest balanceado manteve a maior acurácia geral.
- O LightGBM balanceado apresentou a maior queda de acurácia.
- A Regressão Logística balanceada apresentou o maior recall para a classe `Crítica`, porém com muitos falsos positivos.
- O SVM balanceado apresentou desempenho intermediário, equilibrando melhor acurácia geral e detecção de classes minoritárias.

O SVM conseguiu melhorar significativamente o recall da classe `Atenção`, além de passar a detectar parte das amostras `Críticas`, reduzindo parcialmente o colapso de classe observado anteriormente. Entretanto, ainda apresentou dificuldades relevantes na separação da classe `Boa`, além de baixa precision para `Crítica`.

Os resultados obtidos reforçam a importância da análise conjunta de múltiplas métricas — especialmente recall, precision, F1-score e matriz de confusão — em problemas de classificação desbalanceada aplicados ao contexto ambiental.

Como continuidade metodológica do projeto AquaSense, os próximos experimentos poderão explorar:
- novas variáveis ambientais;
- técnicas adicionais de balanceamento;
- ajuste de hiperparâmetros;
- modelos não lineares mais robustos.

Essas abordagens podem melhorar a detecção de amostras críticas sem aumentar excessivamente os falsos positivos.